# Sinhala Document OCR — End-to-End Colab Pipeline

This notebook runs the full MSc-project pipeline on Google Colab (GPU):
1. Mount Google Drive
2. Get the project + install dependencies
3. Generate synthetic Sinhala line data
4. Train the CRNN recognizer (CNN + BiLSTM + CTC)
5. Evaluate (CER / WER)
6. Run an inference demo

> Tip: set **Runtime > Change runtime type > T4 GPU** before training.

## 1. Mount Google Drive (optional, for saving checkpoints/data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the project
Either clone your repository or upload the folder. Update `PROJECT_DIR` accordingly.

In [ ]:
import os, sys

# Option A: clone from GitHub (replace with your repo URL)
# !git clone https://github.com/<you>/sinhala-document-ocr.git
# PROJECT_DIR = '/content/sinhala-document-ocr'

# Option B: use a copy synced via Google Drive
PROJECT_DIR = '/content/drive/MyDrive/sinhala-document-ocr'

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('Working dir:', os.getcwd())

## 3. Install dependencies
Colab already ships torch, torchvision, numpy, pillow, opencv-python, matplotlib and scikit-learn, so we only add the few extras.

In [ ]:
!pip -q install albumentations editdistance jiwer pyyaml tqdm

## 4. Generate synthetic Sinhala data
On Colab (Linux) install a Sinhala font and point the config at it, or rely on the font list in `configs/default.yaml`.

In [ ]:
# Install Noto Sans Sinhala on Colab so rendering works
!apt-get -qq install -y fonts-noto-sinhala >/dev/null 2>&1 || true
!python scripts/generate_data.py --config configs/default.yaml --num 2000

## 5. Train the CRNN recognizer
Use a few epochs for a quick run; increase `train.epochs` for real training.

In [ ]:
!python -m src.recognition.train --config configs/default.yaml train.epochs=5 train.batch_size=64

## 6. Evaluate on the held-out test split (CER / WER + CPU timing)

In [ ]:
!python -m src.evaluation.metrics \
    --checkpoint models/crnn_best.pth \
    --charset models/charset.json \
    --labels data/synthetic/test_labels.txt --device cpu

## 7. Inference demo on a single line image

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt
from src.charset import Charset
from src.recognition.model import build_crnn
from src.recognition.predict import predict_image
from src.utils.common import load_config, load_checkpoint, get_device

cfg = load_config('configs/default.yaml')
charset = Charset.load('models/charset.json')
device = get_device('cpu')
model = build_crnn(charset.num_classes, cfg['model'], in_channels=cfg['image']['channels']).to(device)
load_checkpoint('models/crnn_best.pth', model, map_location='cpu')
model.eval()

sample = sorted(glob.glob('data/synthetic/images/*.png'))[0]
text = predict_image(model, charset, sample, cfg['image']['height'],
                     cfg['image']['max_width'], cfg['image']['channels'], device)
plt.figure(figsize=(8, 2)); plt.imshow(Image.open(sample)); plt.axis('off')
plt.title('prediction: ' + text); plt.show()